In [1]:
# ANÁLISIS ESTADÍSTICO DE CRÉDITO HIPOTECARIO ASEGURADO
# 1. Instalación de paqueterias
!pip install pandas matplotlib seaborn scikit-learn plotly -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Configurar visualizaciones
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

In [2]:
from google.colab import files
import pandas as pd
import os

print("Sube el archivo CSV:")
# uploaded = files.upload()  # Descomentar para subir archivo

df_loaded = False

# Try to load from the current directory
for filename in os.listdir('.'):
    if filename.endswith('.csv'):
        try:
            df = pd.read_csv(filename)
            print(f" Archivo cargado: {filename}")
            df_loaded = True
            break
        except Exception as e:
            print(f"Error al cargar {filename}: {e}")

# If no CSV found in current directory, try to load a sample file
if not df_loaded:
    print("No se encontró ningún archivo CSV en el directorio actual. Intentando cargar un archivo de ejemplo...")
    try:
        # Use a common sample CSV that's likely to be there
        sample_csv_path = '/content/51_Informacion_estadistica_Credito_Vivienda_Datos_Credito_Asegurado_ok.csv' # You can change this to another sample CSV if needed
        df = pd.read_csv(sample_csv_path)
        print(f" Archivo de ejemplo '{os.path.basename(sample_csv_path)}' cargado.")
        df_loaded = True
    except Exception as e:
        print(f"Error al cargar el archivo de ejemplo: {e}")
        print("No se pudo cargar ningún DataFrame. Por favor, sube un archivo CSV o verifica la ruta.")
        # Initialize df as an empty DataFrame to avoid NameError in subsequent lines
        df = pd.DataFrame()

# 3. EXPLORACIÓN INICIAL DE DATOS
print("\n" + "="*60)
print("3. EXPLORACIÓN INICIAL DE DATOS")
print("="*60)

if not df_loaded or df.empty:
    print("No hay datos cargados para realizar la exploración inicial.")
else:
    print(f"\n Dimensiones del dataset: {df.shape[0]} filas, {df.shape[1]} columnas")

    print("\n Tipos de datos:")
    print(df.dtypes.value_counts())

    print("\n Primeras 5 filas:")
    display(df.head())

    print("\n Estadísticas descriptivas (variables numéricas):")
    display(df.describe())

    print("\n Valores nulos por columna:")
    print(df.isnull().sum())


Sube el archivo CSV:
No se encontró ningún archivo CSV en el directorio actual. Intentando cargar un archivo de ejemplo...
Error al cargar el archivo de ejemplo: [Errno 2] No such file or directory: '/content/51_Informacion_estadistica_Credito_Vivienda_Datos_Credito_Asegurado_ok.csv'
No se pudo cargar ningún DataFrame. Por favor, sube un archivo CSV o verifica la ruta.

3. EXPLORACIÓN INICIAL DE DATOS
No hay datos cargados para realizar la exploración inicial.


In [3]:
# 4. LIMPIEZA DE DATOS
# Ver valores negativos en primas (pueden ser ajustes contables)
if 'prima_emitida' in df.columns:
    negativos = df[df['prima_emitida'] < 0].shape[0]
    print(f" Registros con prima_emitida negativa: {negativos}")

    # Opcional: tratar valores negativos
    df['prima_emitida_abs'] = df['prima_emitida'].abs()

# Verificar datos inconsistentes en monto_enganche
if 'monto_enganche' in df.columns:
    enganche_mayor = df[df['monto_enganche'] > df['monto_credito']].shape[0]
    print(f" Registros con enganche > crédito: {enganche_mayor}")

In [4]:
# 5. ANÁLISIS UNIVARIADO
# Análisis de variables categóricas
categoricas = ['clave_administrador', 'apoyo_financiamiento', 'tipo_tasa',
               'tipo_cartera', 'moneda', 'frecuencia_pago']

for col in categoricas:
    if col in df.columns:
        print(f"\n {col}:")
        print(df[col].value_counts().head(5))

# Distribución del plazo
if 'plazo_credito' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Histograma
    axes[0].hist(df['plazo_credito'].dropna(), bins=30, edgecolor='black', alpha=0.7)
    axes[0].set_title('Distribución del Plazo de Crédito (meses)')
    axes[0].set_xlabel('Plazo (meses)')
    axes[0].set_ylabel('Frecuencia')

    # Boxplot
    axes[1].boxplot(df['plazo_credito'].dropna(), vert=False)
    axes[1].set_title('Boxplot del Plazo de Crédito')
    axes[1].set_xlabel('Plazo (meses)')

    plt.tight_layout()
    plt.show()

    print(f" Estadísticas del plazo:")
    print(f"   Media: {df['plazo_credito'].mean():.0f} meses")
    print(f"   Mediana: {df['plazo_credito'].median():.0f} meses")
    print(f"   Moda: {df['plazo_credito'].mode().iloc[0] if not df['plazo_credito'].mode().empty else 'N/A'} meses")

# Distribución del porcentaje de cobertura
if 'porcentaje_cobertura' in df.columns:
    plt.figure(figsize=(10, 5))
    cobertura_counts = df['porcentaje_cobertura'].value_counts().sort_index()
    plt.bar(cobertura_counts.index.astype(str), cobertura_counts.values, edgecolor='black')
    plt.title('Distribución del Porcentaje de Cobertura')
    plt.xlabel('Porcentaje de Cobertura (%)')
    plt.ylabel('Número de registros')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    print(f" Cobertura más común: {df['porcentaje_cobertura'].mode().iloc[0]}%")


In [5]:
# 6. ANÁLISIS BIVARIADO
# Relación entre tipo de cartera y monto de crédito
if 'tipo_cartera' in df.columns and 'monto_credito' in df.columns:
    fig, ax = plt.subplots(figsize=(12, 6))

    tipo_cartera_order = df.groupby('tipo_cartera')['monto_credito'].median().sort_values().index
    sns.boxplot(data=df, x='tipo_cartera', y='monto_credito', order=tipo_cartera_order, ax=ax)
    ax.set_title('Distribución del Monto de Crédito por Tipo de Cartera')
    ax.set_ylabel('Monto de Crédito')
    ax.set_xlabel('Tipo de Cartera')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

    print("\n Estadísticas por tipo de cartera:")
    for cartera in df['tipo_cartera'].unique():
        datos = df[df['tipo_cartera'] == cartera]['monto_credito']
        print(f"   {cartera}: Media=${datos.mean():,.0f}, Mediana=${datos.median():,.0f}")

# Relación entre plazo y monto de crédito
if 'plazo_credito' in df.columns and 'monto_credito' in df.columns:
    fig, ax = plt.subplots(figsize=(10, 6))

    # Agrupar por plazo redondeado
    df['plazo_redondeado'] = (df['plazo_credito'] / 12).round().astype('Int64') * 12
    plazo_stats = df.groupby('plazo_redondeado')['monto_credito'].agg(['mean', 'median', 'count']).reset_index()

    ax.bar(plazo_stats['plazo_redondeado'].astype(str), plazo_stats['median'],
           edgecolor='black', alpha=0.7)
    ax.set_title('Monto de Crédito Mediano por Plazo (años)')
    ax.set_xlabel('Plazo (años)')
    ax.set_ylabel('Monto de Crédito Mediano')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


In [6]:
# 7. ANÁLISIS POR INSTITUCIONES
if 'clave_administrador' in df.columns:
    # Top 10 instituciones por número de créditos
    top_instituciones = df.groupby('clave_administrador')['numero_creditos'].sum().sort_values(ascending=False).head(10)

    fig, ax = plt.subplots(figsize=(12, 6))
    top_instituciones.plot(kind='barh', ax=ax)
    ax.set_title('Top 10 Instituciones por Número Total de Créditos')
    ax.set_xlabel('Número de Créditos')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

    print(f"\n Top 5 instituciones:")
    for i, (inst, creditos) in enumerate(top_instituciones.head(5).items(), 1):
        print(f"   {i}. {inst}: {creditos:,.0f} créditos")

# 8. ANÁLISIS DE CORRELACIÓN
print("\n" + "="*60)
print("8. ANÁLISIS DE CORRELACIÓN")
print("="*60)

# Seleccionar variables numéricas para correlación
num_cols = ['plazo_credito', 'porcentaje_cobertura', 'numero_creditos',
            'monto_enganche', 'monto_credito', 'prima_emitida',
            'prima_devengada', 'saldo_principal']

num_cols_existentes = [col for col in num_cols if col in df.columns]

if len(num_cols_existentes) > 1:
    corr_matrix = df[num_cols_existentes].corr()

    fig, ax = plt.subplots(figsize=(10, 8))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
                square=True, linewidths=0.5, ax=ax)
    ax.set_title('Matriz de Correlación entre Variables Numéricas')
    plt.tight_layout()
    plt.show()

    print("\n Correlaciones más fuertes:")
    corr_pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j],
                              corr_matrix.iloc[i, j]))
    corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)

    for col1, col2, corr in corr_pairs[:5]:
        print(f"   {col1} ↔ {col2}: {corr:.3f}")


8. ANÁLISIS DE CORRELACIÓN


In [7]:
# 9. ANÁLISIS TEMPORAL
if 'anio' in df.columns and 'monto_credito' in df.columns:
    anio_stats = df.groupby('anio').agg({
        'monto_credito': ['sum', 'mean'],
        'numero_creditos': 'sum',
        'saldo_principal': 'sum'
    }).round(0)

    anio_stats.columns = ['Monto_Total', 'Monto_Promedio', 'Total_Creditos', 'Saldo_Principal']
    anio_stats = anio_stats.reset_index()

    print("\n Evolución anual:")
    display(anio_stats)

    # Gráfico de evolución
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(anio_stats['anio'], anio_stats['Monto_Total'] / 1e9, 'o-', linewidth=2, markersize=8)
    axes[0].set_title('Evolución del Monto Total de Créditos')
    axes[0].set_xlabel('Año')
    axes[0].set_ylabel('Monto Total (Miles de Millones)')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(anio_stats['anio'], anio_stats['Total_Creditos'], 's-', linewidth=2, markersize=8, color='green')
    axes[1].set_title('Evolución del Número Total de Créditos')
    axes[1].set_xlabel('Año')
    axes[1].set_ylabel('Número de Créditos')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


In [8]:
# 10. ANÁLISIS DE RIESGO - RATIO SALDO/CRÉDITO
if 'saldo_principal' in df.columns and 'monto_credito' in df.columns:
    # Calcular ratio de pago (qué porcentaje del crédito aún se debe)
    df['ratio_saldo'] = df['saldo_principal'] / df['monto_credito']

    # Limitar a valores razonables
    df_ratio_clean = df[(df['ratio_saldo'] >= 0) & (df['ratio_saldo'] <= 1.5)]

    print(f"\n Estadísticas del Ratio Saldo/Crédito:")
    print(f"   Media: {df_ratio_clean['ratio_saldo'].mean():.3f}")
    print(f"   Mediana: {df_ratio_clean['ratio_saldo'].median():.3f}")
    print(f"   Desviación: {df_ratio_clean['ratio_saldo'].std():.3f}")

    # Distribución por tipo de cartera
    if 'tipo_cartera' in df.columns:
        fig, ax = plt.subplots(figsize=(12, 6))
        sns.boxplot(data=df_ratio_clean, x='tipo_cartera', y='ratio_saldo', ax=ax)
        ax.set_title('Distribución del Ratio Saldo/Crédito por Tipo de Cartera')
        ax.set_ylabel('Ratio Saldo/Crédito')
        ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Medio pagado')
        ax.legend()
        plt.tight_layout()
        plt.show()

In [9]:
# 11. CLUSTERIZACIÓN K-MEANS
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Preparar datos para clustering
cluster_cols = ['plazo_credito', 'porcentaje_cobertura', 'monto_credito', 'saldo_principal']
cluster_cols_existentes = [col for col in cluster_cols if col in df.columns]

if len(cluster_cols_existentes) >= 2 and df.shape[0] > 100:
    # Eliminar valores nulos y extremos
    cluster_data = df[cluster_cols_existentes].dropna()
    cluster_data = cluster_data[(cluster_data['monto_credito'] > 0) &
                                 (cluster_data['saldo_principal'] > 0)]

    # Tomar muestra para mejor rendimiento
    if len(cluster_data) > 5000:
        cluster_data = cluster_data.sample(n=5000, random_state=42)

    # Estandarizar
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(cluster_data)

    # Encontrar número óptimo de clusters (método del codo)
    inertias = []
    K_range = range(1, 11)
    for k in K_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(data_scaled)
        inertias.append(kmeans.inertia_)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(K_range, inertias, 'o-')
    ax.set_xlabel('Número de Clusters (k)')
    ax.set_ylabel('Inercia')
    ax.set_title('Método del Codo para Selección de k')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Aplicar k-means con k=3 o 4
    optimal_k = 4
    kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(data_scaled)

    # Agregar clusters al DataFrame
    cluster_data_copy = cluster_data.copy()
    cluster_data_copy['Cluster'] = clusters

    # Visualizar clusters (primeras 2 componentes)
    from sklearn.decomposition import PCA

    pca = PCA(n_components=2)
    data_pca = pca.fit_transform(data_scaled)

    fig, ax = plt.subplots(figsize=(10, 8))
    scatter = ax.scatter(data_pca[:, 0], data_pca[:, 1], c=clusters, cmap='viridis', alpha=0.6, s=20)
    ax.set_title(f'Visualización de Clusters (k={optimal_k})')
    ax.set_xlabel('Componente Principal 1')
    ax.set_ylabel('Componente Principal 2')
    plt.colorbar(scatter, label='Cluster')
    plt.tight_layout()
    plt.show()

    print("\n Características de cada cluster:")
    cluster_summary = cluster_data_copy.groupby('Cluster').agg({
        'plazo_credito': ['mean', 'count'],
        'monto_credito': ['mean', 'median'],
        'saldo_principal': 'mean'
    }).round(0)
    display(cluster_summary)

    print("\n Interpretación de clusters:")
    for i in range(optimal_k):
        cluster_data_i = cluster_data_copy[cluster_data_copy['Cluster'] == i]
        plazo_medio = cluster_data_i['plazo_credito'].mean()
        monto_medio = cluster_data_i['monto_credito'].mean()
        print(f"   Cluster {i}: {len(cluster_data_i)} créditos, "
              f"plazo medio={plazo_medio:.0f} meses, "
              f"monto medio=${monto_medio:,.0f}")